# v16.6 — aturan episode AF dengan toleransi celah: dipilih di data lama, divalidasi di SHDB-AF, afpdb, incartdb

Rancangan dikunci sebelum sinyal data validasi dibaca: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v16.6_praregistrasi.log`,
termasuk amandemen A). Logika di `celah.py`; metrik episode dari v16.5, rantai prediksi dari v16.4.

- **Tahap A**: grid durasi minimum {30, 45, 60, 90} s × celah {0, 1, 2} window untuk D-1 di data v16.5; pemilihan dikunci.
- **Tahap B**: konfigurasi terpilih diuji sekali di data yang belum pernah dipakai.

In [1]:
import gc, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import celah as C
import sys; sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
t0 = time.time()
R = {}
for ds in C.VALIDASI:
    C.jalankan(ds)                     # dilewati bila cache per rekaman lengkap
D = C.E.muat_semua()
print("paritas g = 0, 30 s dengan v16.5:", C.cek_paritas(D))

paritas g = 0, 30 s dengan v16.5: True


## 1. Tahap A: grid di data pengembangan (D-1)

In [2]:
G = C.grid(D)
rows = []
for (sp, g), h in G.items():
    rows.append({"span": sp, "celah": g, **{f"palsu {x}": round(h[x]["palsu_per_jam"], 3) for x in C.E_NEG},
                 **{f"Se30 {x}": round(h[x]["se_episode_30"], 3) for x in C.E_AF},
                 "Se30 min": round(min(h[x]["se_episode_30"] for x in C.E_AF), 3)})
tabA = pd.DataFrame(rows); display(tabA)
(sp, g), ok = C.pilih(G)
print("TERPILIH: span", sp, "s, celah", g, "| memenuhi batas alarm palsu di data pengembangan:", ok)
R["tahapA"] = dict(grid=rows, terpilih=dict(span=sp, celah=g, lolos_batas=ok))
ref = {ds: C.nilai(d, d["v31"], sp, g) for ds, d in D.items()}
ref30 = {ds: C.nilai(d, d["v31"], 30, 0) for ds, d in D.items()}
R["tahapA"]["konteks"] = {ds: {"D-1 terpilih": {k: G[(sp, g)][ds][k] for k in ("palsu_per_jam", "se_episode_30", "se_durasi", "ppv_durasi")},
                              "v3.1 terpilih": {k: ref[ds][k] for k in ("palsu_per_jam", "se_episode_30", "se_durasi", "ppv_durasi")},
                              "v3.1 30 s": {k: ref30[ds][k] for k in ("palsu_per_jam", "se_episode_30", "se_durasi", "ppv_durasi")}}
                         for ds in D}
display(pd.DataFrame({ds: {f"{s} {k}": round(v, 3) for s, m in x.items() for k, v in m.items() if k in ("palsu_per_jam", "se_episode_30")}
                      for ds, x in R["tahapA"]["konteks"].items()}))
del D, ref, ref30; gc.collect()

  span 30 g 0: palsu NSRDB 1.234 EDB 0.176 latihan 0.133 | Se30 min 0.911


  span 30 g 1: palsu NSRDB 3.622 EDB 0.358 latihan 1.131 | Se30 min 0.958


  span 30 g 2: palsu NSRDB 8.102 EDB 0.575 latihan 1.630 | Se30 min 0.968


  span 45 g 0: palsu NSRDB 0.208 EDB 0.051 latihan 0.000 | Se30 min 0.726


  span 45 g 1: palsu NSRDB 1.327 EDB 0.182 latihan 0.333 | Se30 min 0.773


  span 45 g 2: palsu NSRDB 5.382 EDB 0.319 latihan 0.499 | Se30 min 0.864


... (6 outputs left out)

,span,celah,palsu NSRDB,palsu EDB,palsu Latihan fisik,Se30 AFDB,Se30 LTAFDB,Se30 MIT-BIH,Se30 CPSC 2021,Se30 min
0,30,0,1.234,0.176,0.133,0.927,0.941,0.977,0.911,0.911
1,30,1,3.622,0.358,1.131,0.961,0.958,0.977,0.960,0.958
2,30,2,8.102,0.575,1.630,0.976,0.968,0.977,0.976,0.968
3,45,0,0.208,0.051,0.000,0.799,0.726,0.773,0.728,0.726
4,45,1,1.327,0.182,0.333,0.882,0.790,0.773,0.827,0.773
5,45,2,5.382,0.319,0.499,0.929,0.871,0.864,0.918,0.864
6,60,0,0.039,0.006,0.000,0.692,0.592,0.682,0.604,0.592
7,60,1,0.497,0.068,0.266,0.799,0.684,0.682,0.744,0.682
8,60,2,3.500,0.182,0.499,0.875,0.801,0.841,0.868,0.801
9,90,0,0.002,0.000,0.000,0.523,0.444,0.568,0.452,0.444


TERPILIH: span 45 s, celah 0 | memenuhi batas alarm palsu di data pengembangan: True


,AFDB,LTAFDB,MIT-BIH,CPSC 2021,NSRDB,EDB,Latihan fisik
D-1 terpilih palsu_per_jam,0.039,0.216,0.000,0.358,0.208,0.051,0.000
D-1 terpilih se_episode_30,0.799,0.726,0.773,0.728,NaN,1.000,NaN
v3.1 terpilih palsu_per_jam,0.744,0.331,1.628,1.224,1.065,0.432,4.890
v3.1 terpilih se_episode_30,0.824,0.685,0.727,0.696,NaN,1.000,NaN
v3.1 30 s palsu_per_jam,1.592,0.942,3.714,2.980,3.569,1.024,8.815
v3.1 30 s se_episode_30,0.947,0.922,0.886,0.892,NaN,1.000,NaN


44

## 2. Tahap B: data validasi (belum pernah dipakai)

In [3]:
V, info = {}, {}
for ds in C.VALIDASI:
    V[ds], info[ds] = C.muat(ds)
R["data"] = info
display(pd.DataFrame(info).T)
SIS = {"D-1 terpilih": ("d1", sp, g), "v3.1 terpilih": ("v31", sp, g), "D-1 30 s": ("d1", 30, 0), "v3.1 30 s": ("v31", 30, 0)}
H = {ds: {s: C.nilai(V[ds], V[ds][k], a, b) for s, (k, a, b) in SIS.items()} for ds in C.VALIDASI}
KOL = ("palsu_per_jam", "se_episode_30", "n_ref_30", "se_episode", "ppv_episode", "se_durasi", "ppv_durasi", "jam")
R["hasil"] = {ds: {s: {k: m[k] for k in KOL} for s, m in h.items()} for ds, h in H.items()}
for ds in C.VALIDASI:
    print("==", ds); display(pd.DataFrame(R["hasil"][ds]).T.round(3))

,rekaman,kosong,window_lolos,dibuang_label,window,window_AF,unit,jam
SHDB-AF,98.0,0.0,1662768.0,3229.0,1659539.0,343286.0,93.0,2304.915278
afpdb,100.0,0.0,35465.0,0.0,35465.0,0.0,100.0,49.256944
incartdb,75.0,0.0,25166.0,16.0,25150.0,739.0,32.0,34.930556
svdb,78.0,0.0,27814.0,2.0,27812.0,0.0,78.0,38.627778


== SHDB-AF


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
D-1 terpilih,0.377,0.700,1648.0,0.351,0.820,0.850,0.928,2304.915
v3.1 terpilih,0.804,0.660,1648.0,0.332,0.749,0.849,0.885,2304.915
D-1 30 s,0.866,0.942,1648.0,0.473,0.725,0.875,0.909,2304.915
v3.1 30 s,1.798,0.865,1648.0,0.434,0.628,0.878,0.851,2304.915


== afpdb


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
D-1 terpilih,0.345,NaN,0.0,NaN,0.0,NaN,0.0,49.257
v3.1 terpilih,0.650,NaN,0.0,NaN,0.0,NaN,0.0,49.257
D-1 30 s,0.893,NaN,0.0,NaN,0.0,NaN,0.0,49.257
v3.1 30 s,2.213,NaN,0.0,NaN,0.0,NaN,0.0,49.257


== incartdb


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
D-1 terpilih,0.200,1.0,4.0,0.667,0.682,0.589,0.854,34.931
v3.1 terpilih,2.720,1.0,4.0,0.667,0.112,0.203,0.080,34.931
D-1 30 s,0.773,1.0,4.0,0.667,0.471,0.674,0.708,34.931
v3.1 30 s,5.983,1.0,4.0,0.667,0.147,0.421,0.112,34.931


== svdb


,palsu_per_jam,se_episode_30,n_ref_30,se_episode,ppv_episode,se_durasi,ppv_durasi,jam
D-1 terpilih,0.932,NaN,0.0,NaN,0.0,NaN,0.0,38.628
v3.1 terpilih,2.977,NaN,0.0,NaN,0.0,NaN,0.0,38.628
D-1 30 s,2.071,NaN,0.0,NaN,0.0,NaN,0.0,38.628
v3.1 30 s,6.368,NaN,0.0,NaN,0.0,NaN,0.0,38.628


## 3. Target, H1, H2 (dikunci)

In [4]:
sel = H["SHDB-AF"]["D-1 terpilih"]
target = dict(se_shdb=sel["se_episode_30"], palsu={ds: H[ds]["D-1 terpilih"]["palsu_per_jam"] for ds in ("SHDB-AF", "afpdb", "incartdb")})
target["se_ok"] = bool(target["se_shdb"] >= C.TARGET_SE)
target["palsu_ok"] = bool(all(v <= C.TARGET_PALSU for v in target["palsu"].values()))
h1 = C.boot_rasio(H["SHDB-AF"]["D-1 terpilih"]["palsu_unit"], H["SHDB-AF"]["v3.1 terpilih"]["palsu_unit"])
h2 = C.boot_rasio(H["SHDB-AF"]["D-1 terpilih"]["se_unit"], H["SHDB-AF"]["v3.1 terpilih"]["se_unit"])
sek_p = C.boot_rasio(H["SHDB-AF"]["D-1 terpilih"]["palsu_unit"], H["SHDB-AF"]["D-1 30 s"]["palsu_unit"])
sek_s = C.boot_rasio(H["SHDB-AF"]["D-1 terpilih"]["se_unit"], H["SHDB-AF"]["D-1 30 s"]["se_unit"])
ci_se = C.boot_rasio(H["SHDB-AF"]["D-1 terpilih"]["se_unit"], H["SHDB-AF"]["D-1 terpilih"]["se_unit"])["ci_a"]
ci_p = {ds: C.boot_rasio(H[ds]["D-1 terpilih"]["palsu_unit"], H[ds]["D-1 terpilih"]["palsu_unit"])["ci_a"] for ds in C.VALIDASI}
k = dict(target_se=target["se_ok"], target_palsu=target["palsu_ok"], H1=bool(h1["p_kurang"] >= 0.95), H2=bool(h2["p_lebih"] > 0.05))
k["semua"] = all(k.values())
R.update(target=target, H1=h1, H2=h2, sekunder=dict(palsu_vs_30=sek_p, se_vs_30=sek_s), ci=dict(se_shdb=ci_se, palsu=ci_p), keputusan=k)
print("target:", json.dumps(target, ensure_ascii=False)); print("CI Se SHDB-AF:", ci_se, "| CI palsu/jam:", ci_p)
print("H1 palsu/jam SHDB-AF D-1 − v3.1:", h1); print("H2 Se episode SHDB-AF D-1 − v3.1:", h2)
print("sekunder D-1 terpilih − D-1 30 s: palsu", sek_p, "| Se", sek_s)
print(json.dumps(k))

target: {"se_shdb": 0.6996359223300971, "palsu": {"SHDB-AF": 0.37745422072033247, "afpdb": 0.34512900042295247, "incartdb": 0.2003976143141153}, "se_ok": false, "palsu_ok": true}
CI Se SHDB-AF: [0.6219761813342076, 0.800775295450866] | CI palsu/jam: {'SHDB-AF': [0.1546339663907259, 0.6590247092610212], 'afpdb': [0.020187293838694435, 0.8336133904614152], 'incartdb': [0.0, 0.557595091545707], 'svdb': [0.3642389148636709, 1.6527991865647322]}
H1 palsu/jam SHDB-AF D-1 − v3.1: {'delta': -0.42604602844524886, 'ci': [-0.8318645032397909, -0.05636292695593153], 'p_lebih': 0.015, 'p_kurang': 0.9845, 'titik_a': 0.3774542207203325, 'ci_a': [0.1546339663907259, 0.6590247092610212], 'titik_b': 0.8035002491655814}
H2 Se episode SHDB-AF D-1 − v3.1: {'delta': 0.03944174757281549, 'ci': [-0.07829059042578759, 0.1463814789481684], 'p_lebih': 0.678, 'p_kurang': 0.315, 'titik_a': 0.6996359223300971, 'ci_a': [0.6219761813342076, 0.800775295450866], 'titik_b': 0.6601941747572816}
sekunder D-1 terpilih − D-

## 4. Alarm non-AF per pita detak dan per subjek (deskriptif)

In [5]:
PITA = ((0, 100, "< 100"), (100, 110, "100–110"), (110, 120, "110–120"), (120, 999, "≥ 120"))
pita = {}
for ds in ("SHDB-AF", "afpdb", "incartdb", "svdb"):
    d = V[ds]; non = d["y2"] == 0
    for s, (kk, a, b) in SIS.items():
        reps = d[kk] if d[kk].ndim == 2 else d[kk][None]
        al = C.aturan_celah(d["rec"], d["start"], reps[0], d["langkah"], a, b)
        pita.setdefault(ds, {})[s] = {lab: dict(n=int((non & (d["hr"] >= lo) & (d["hr"] < hi)).sum()),
                                               alarm=float(al[non & (d["hr"] >= lo) & (d["hr"] < hi)].mean()) if (non & (d["hr"] >= lo) & (d["hr"] < hi)).any() else None)
                                      for lo, hi, lab in PITA}
R["pita"] = pita
display(pd.DataFrame({(ds, s): {b: (f"{v['alarm']*100:.2f}% (n={v['n']})" if v["alarm"] is not None else "–") for b, v in p.items()}
                      for ds, x in pita.items() for s, p in x.items()}).T)
pu = H["SHDB-AF"]["D-1 terpilih"]["palsu_unit"]
R["shdb_per_subjek"] = {u: dict(palsu=v[0], jam=v[1], se=H["SHDB-AF"]["D-1 terpilih"]["se_unit"].get(u, [0, 0])) for u, v in pu.items()}
top = sorted(pu, key=lambda u: -pu[u][0] / max(pu[u][1], 1e-9))[:8]
print("SHDB-AF subjek dengan alarm palsu/jam tertinggi:", {u: round(pu[u][0] / pu[u][1], 2) for u in top})

< 100          100–110          110–120  \
SHDB-AF  D-1 terpilih   1.39% (n=1270492)  6.55% (n=32504)  12.99% (n=9277)   
         v3.1 terpilih  2.14% (n=1270492)  7.43% (n=32504)  34.72% (n=9277)   
         D-1 30 s       1.85% (n=1270492)  7.21% (n=32504)  14.24% (n=9277)   
         v3.1 30 s      3.09% (n=1270492)  8.35% (n=32504)  39.19% (n=9277)   
afpdb    D-1 terpilih     0.31% (n=31263)   3.55% (n=3125)    0.00% (n=944)   
         v3.1 terpilih    0.80% (n=31263)   0.90% (n=3125)    9.43% (n=944)   
         D-1 30 s         0.63% (n=31263)   5.12% (n=3125)    0.74% (n=944)   
         v3.1 30 s        1.77% (n=31263)   2.59% (n=3125)   15.57% (n=944)   
incartdb D-1 terpilih     0.28% (n=22499)    0.24% (n=834)    0.46% (n=435)   
         v3.1 terpilih    3.14% (n=22499)   13.55% (n=834)   61.15% (n=435)   
         D-1 30 s         0.70% (n=22499)    2.40% (n=834)    0.69% (n=435)   
         v3.1 30 s        5.57% (n=22499)   15.59% (n=834)   69.43% (n=435)   
svdb     D-1 terpilih     2.59% (n=24557)   9.94% (n=1700)   10.65% (n=845)   
         v3.1 terpilih    4.33% (n=24557)  16.71% (n=1700)   50.53% (n=845)   
         D-1 30 s         3.37% (n=24557)  10.82% (n=1700)   14.32% (n=845)   
         v3.1 30 s        6.49% (n=24557)  22.88% (n=1700)   63.91% (n=845)   

                                  ≥ 120  
SHDB-AF  D-1 terpilih   26.98% (n=3980)  
         v3.1 terpilih  87.91% (n=3980)  
         D-1 30 s       34.37% (n=3980)  
         v3.1 30 s      94.65% (n=3980)  
afpdb    D-1 terpilih     0.00% (n=133)  
         v3.1 terpilih   87.97% (n=133)  
         D-1 30 s         0.75% (n=133)  
         v3.1 30 s       98.50% (n=133)  
incartdb D-1 terpilih     0.00% (n=643)  
         v3.1 terpilih   87.25% (n=643)  
         D-1 30 s         0.00% (n=643)  
         v3.1 30 s       94.25% (n=643)  
svdb     D-1 terpilih     5.77% (n=710)  
         v3.1 terpilih   95.63% (n=710)  
         D-1 30 s         7.89% (n=710)  
         v3.1 30 s       96.90% (n=710)

SHDB-AF subjek dengan alarm palsu/jam tertinggi: {'5354388': 6.71, '5196228': 4.87, '5645016': 2.82, '5668218': 2.64, '5633247': 2.62, '5113680': 2.3, '739914': 1.89, '2416281': 1.88}


## 5. Kesimpulan

In [6]:
h = R["hasil"]
print(f"Tahap A memilih span {sp} s, celah {g} window (data pengembangan: alarm palsu ≤ 1,0/jam di NSRDB, EDB, latihan: {ok}).")
for ds in C.VALIDASI:
    m, r = h[ds]["D-1 terpilih"], h[ds]["v3.1 terpilih"]
    print(f"{ds}: D-1 palsu/jam {m['palsu_per_jam']:.2f} (v3.1 {r['palsu_per_jam']:.2f}); Se episode ≥30 s {m['se_episode_30']:.3f} (v3.1 {r['se_episode_30']:.3f}); "
          f"Se durasi {m['se_durasi']:.3f}; {m['jam']:.0f} jam")
print(f"Target Se SHDB-AF ≥ 0,90: {'lolos' if k['target_se'] else 'GAGAL'}; alarm palsu ≤ 1,0/jam: {'lolos' if k['target_palsu'] else 'GAGAL'}; "
      f"H1 {'lolos' if k['H1'] else 'GAGAL'} (P = {h1['p_kurang']:.3f}); H2 {'lolos' if k['H2'] else 'GAGAL'} (P(Δ>0) = {h2['p_lebih']:.3f}).")
print("Keputusan:", "D-1 + konfigurasi terpilih = sistem kandidat terbaik untuk uji perangkat" if k["semua"] else "ada kriteria yang gagal (lihat README)")
R["lingkungan"] = Ev._lingkungan(); R["durasi_menit"] = (time.time() - t0) / 60
C.simpan_json(R, C.CACHE / "ringkasan.json"); print("tersimpan")

Tahap A memilih span 45 s, celah 0 window (data pengembangan: alarm palsu ≤ 1,0/jam di NSRDB, EDB, latihan: True).
SHDB-AF: D-1 palsu/jam 0.38 (v3.1 0.80); Se episode ≥30 s 0.700 (v3.1 0.660); Se durasi 0.850; 2305 jam
afpdb: D-1 palsu/jam 0.35 (v3.1 0.65); Se episode ≥30 s nan (v3.1 nan); Se durasi nan; 49 jam
incartdb: D-1 palsu/jam 0.20 (v3.1 2.72); Se episode ≥30 s 1.000 (v3.1 1.000); Se durasi 0.589; 35 jam
svdb: D-1 palsu/jam 0.93 (v3.1 2.98); Se episode ≥30 s nan (v3.1 nan); Se durasi nan; 39 jam
Target Se SHDB-AF ≥ 0,90: GAGAL; alarm palsu ≤ 1,0/jam: lolos; H1 lolos (P = 0.985); H2 lolos (P(Δ>0) = 0.678).
Keputusan: ada kriteria yang gagal (lihat README)
tersimpan
